In [3]:
"""
This script downloads launch data from The Space Devs Launch Library API
and saves it into a CSV file for analysis.

If the script stops because of the API rate limit, rerun it later.
It will resume from where the CSV left off.
"""

import requests
import pandas as pd
import time
import os

BASE_URL = "https://ll.thespacedevs.com/2.3.0/launches/"
CSV_FILE = "space_devs_launches_2010_2026.csv"

# Load existing CSV if it exists
if os.path.exists(CSV_FILE):
    df = pd.read_csv(CSV_FILE)
    start_offset = len(df)
    print(f"Found existing CSV with {start_offset} rows. Resuming from there.")
else:
    df = pd.DataFrame()
    start_offset = 0
    print("No existing CSV found. Starting from the beginning.")

params = { # DO NOT CHANGE THESE MIDWAY THROUGH
    "limit": 100,
    "offset": start_offset,
    "mode": "normal",
    "ordering": "net",
    "format": "json",
    "net__gte": "2010-01-01T00:00:00Z",
    "net__lte": "2026-12-31T23:59:59Z",
    "include_suborbital": "false"
}

url = BASE_URL

while url:
    response = requests.get(
        url,
        params=params if url == BASE_URL else None,
        timeout=30
    )

    if response.status_code == 429:
        print("Rate limit hit. Progress has already been saved.")
        break

    response.raise_for_status()

    data = response.json()

    page_df = pd.json_normalize(data["results"])

    df = pd.concat([df, page_df], ignore_index=True)

    if "id" in df.columns:
        df = df.drop_duplicates(subset="id")

    df.to_csv(CSV_FILE, index=False)

    print(f"Saved {len(df)} of {data['count']} launches.")

    url = data["next"]

    time.sleep(5)

print(f"Finished this run. CSV currently has {len(df)} rows.")

Found existing CSV with 1400 rows. Resuming from there.
Saved 1500 of 2469 launches.
Saved 1600 of 2469 launches.
Saved 1700 of 2469 launches.
Saved 1800 of 2469 launches.
Saved 1900 of 2469 launches.
Saved 2000 of 2469 launches.
Saved 2100 of 2469 launches.
Saved 2200 of 2469 launches.
Saved 2300 of 2469 launches.
Saved 2400 of 2469 launches.


C:\Users\Kura\AppData\Local\Temp\ipykernel_36216\368615881.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, page_df], ignore_index=True)


Saved 2469 of 2469 launches.
Finished this run. CSV currently has 2469 rows.
